In [ ]:
import fitz  # PyMuPDF
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
def detect_checkboxes(image, scale_x=1.0, scale_y=1.0):
    checkboxes = []
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if 10 <= w <= 30 and 10 <= h <= 30:
            aspect_ratio = w / h if h > 0 else 0
            if 0.8 <= aspect_ratio <= 1.2:
                x0, y0 = x * scale_x, y * scale_y
                x1, y1 = (x + w) * scale_x, (y + h) * scale_y
                checkboxes.append([round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)])
    return checkboxes

# Load a single page image from PDF and detect checkboxes, then visualize with matplotlib
def visualize_checkboxes(pdf_path, page_num=0, dpi=300):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    img_cv = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
    scale_x = page.rect.width / pix.w
    scale_y = page.rect.height / pix.h

    checkboxes = detect_checkboxes(img_cv, scale_x, scale_y)

    # Plot
    fig, ax = plt.subplots(figsize=(pix.w / dpi * 2.54, pix.h / dpi * 2.54), dpi=100)  # size in inches
    ax.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))

    for bbox in checkboxes:
        x0, y0, x1, y1 = bbox
        width = x1 - x0
        height = y1 - y0
        rect = Rectangle((x0, y0), width, height, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

    ax.axis('off')
    plt.title(f"Detected Checkboxes on page {page_num}")
    plt.show()
    doc.close()

# Example usage:
# Replace 'Claim_Form.pdf' with the path to your PDF file
# visualize_c

In [ ]:
visualize_checkboxes('Claim_Form.pdf', page_num=0)
